## Setup

In [1]:
# !git clone https://github.com/CTLab-ITMO/self-expanding-nets -b freeze-layers

In [2]:
# !pip install -U -e ./self-expanding-nets/ --force-reinstall

## Imports

In [3]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from senmodel.metrics.edge_finder import EdgeFinder
from senmodel.model.utils import *
from senmodel.metrics.nonlinearity_metrics import *

from tqdm import tqdm


SEED = 8642

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

device = 'cpu' #torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

'cpu'

## Data

In [4]:
BATCH_SIZE = 64

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Lambda(lambda x: x.view(-1)),
     ]
)

train_dataset = datasets.FashionMNIST(root='./data', train=True,
                                      download=True, transform=transform)
val_dataset = datasets.FashionMNIST(root='./data', train=False,
                                    download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Model

In [5]:
# Define the model
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28):
        super(SimpleFCN, self).__init__()
        self.fc1 = nn.Linear(input_size, 10)

    def forward(self, x):
        x = self.fc1(x)
        return x

In [6]:
device

'cpu'

In [7]:
train_dataset[0][0].shape

torch.Size([784])

In [8]:
model = SimpleFCN(input_size=784).to(device)
input_tensor = torch.randn(1, 784).to(device)
output = model(input_tensor)
output

tensor([[ 0.2830, -1.0584,  0.9886,  1.3175, -0.0385, -0.3542, -0.2372, -0.3732,
          0.4803,  1.0001]], grad_fn=<AddmmBackward0>)

## Train loop

In [9]:
import time


def train_sparse_recursive(model, train_loader, val_loader, num_epochs, metric, edge_replacement_func=None,
                        window_size=3, threshold=0.1, lr=5e-4, choose_threshold=0.3, aggregation_mode='mean'):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    replace_epoch = [0]
    val_losses = []
    len_choose = get_model_last_layer(model).count_replaces

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        train_loss = 0
        for i, (inputs, targets) in enumerate(tqdm(train_loader)):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            loss.backward()

            # if len(len_choose) > 3 and i > window_size:
            #     freeze_all_but_last(model)

            optimizer.step()
            train_loss += loss.item()
        
        # if len(replace_epoch) > 1:
        #     for g in optimizer.param_groups:
        #         g['lr'] *= 0.9

        train_loss /= len(train_loader)
        train_time = time.time() - t0

        model.eval()
        val_loss = 0
        all_targets = []
        all_preds = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                all_targets.extend(targets.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        val_loss /= len(val_loader)
        val_accuracy = accuracy_score(all_targets, all_preds)

        print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")
        new_l = dict()
        # if edge_replacement_func and epoch % 8 == 0 and epoch != 0:
        #     new_l = edge_replacement_func(model, optimizer, val_loader, metric)
        val_losses.append(val_loss)
        if len(val_losses) > window_size and epoch - replace_epoch[-1] > 8:
            recent_changes = [abs(val_losses[i] - val_losses[i - 1]) for i in range(-window_size, 0)]
            avg_change = sum(recent_changes) / window_size
            if avg_change < threshold:
                len_ch = len_choose[-1]
                # len_ch = get_model_last_layer(model).weight_values.shape[0]

                new_l = edge_replacement_func(model, optimizer, val_loader, metric, choose_threshold, aggregation_mode, len_ch)
                len_choose = get_model_last_layer(model).count_replaces
                replace_epoch += [epoch]
                # if len(replace_epoch) == 2:
                #     for g in optimizer.param_groups:
                #         g['lr'] *= 200
            print(torch.unique(get_model_last_layer(model).weight_indices[0]))

        print(new_l)
        params_amount = get_params_amount(model.fc1)
        zero_params_amount = get_zero_params_amount(model.fc1)
        wandb.log({'val loss': val_loss, 'val accuracy': val_accuracy,
                    'train loss': train_loss, 'params amount': params_amount,
                      'zero params amount': zero_params_amount, 'train time': train_time,
                        'params ratio': (params_amount - zero_params_amount) / params_amount,
                          'lr': optimizer.param_groups[0]['lr']} | new_l)


def edge_replacement_func_new_layer(model, optim, val_loader, metric, choose_threshold, aggregation_mode='mean', len_choose=None):
    layer = get_model_last_layer(model)
    ef = EdgeFinder(metric, val_loader, device, aggregation_mode)
    vals = ef.calculate_edge_metric_for_dataloader(model, len_choose, False)
    print("Edge metrics:", vals, max(vals, default=0), sum(vals))
    chosen_edges = ef.choose_edges_threshold(model, choose_threshold, len_choose)
    print("Chosen edges:", chosen_edges, len(chosen_edges[0]))
    layer.replace_many(*chosen_edges)

    if len(chosen_edges[0]) > 0:
        optim.add_param_group({'params': layer.embed_linears[-1].weight_values})
        # optim.add_param_group({'params': layer.weight_values})
    else:
        print("Empty metric")

    return {'max': max(vals, default=0), 'sum': sum(vals), 'len': len(vals), 'len_choose': layer.count_replaces[-1]}

## Training

In [10]:
criterion = nn.CrossEntropyLoss()
metrics = [
    MagnitudeL2Metric(criterion),
    SNIPMetric(criterion),
    # GradientMeanEdgeMetric(criterion),
    PerturbationSensitivityEdgeMetric(criterion),
]

hyperparams = {
    "num_epochs": 64,
    "metric": metrics[0],  # Выбранная метрика
    "aggregation_mode": "mean",  # Режим агрегации
    "choose_threshold": 0.2,  # Порог выбора ребер
    "window_size": 5,  # Размер окна для анализа изменения потерь
    "threshold": 0.02,  # Порог для замены ребер
    "lr": 1e-4,  # Скорость обучения
}

In [11]:
sparse_model = convert_dense_to_sparse_network(SimpleFCN(784).to(device)).to(device)
# print_layer_status(sparse_model)

In [12]:
import wandb
wandb.finish()
wandb.init(
    project="self-expanding-nets",
    name="fashionmnist test"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: fedornigretuk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
for name, param in sparse_model.named_parameters():
    print(f"{name}: {'cuda' if param.is_cuda else 'cpu'}")

fc1.weight_values: cpu
fc1.bias_values: cpu


In [14]:
train_sparse_recursive(sparse_model, train_loader, val_loader,
                       edge_replacement_func=edge_replacement_func_new_layer,
                       **hyperparams)

100%|██████████| 938/938 [00:12<00:00, 76.88it/s]


Epoch 1/64, Train Loss: 1.1765, Val Loss: 0.8388, Val Accuracy: 0.7278
{}


100%|██████████| 938/938 [00:11<00:00, 81.91it/s]


Epoch 2/64, Train Loss: 0.7416, Val Loss: 0.6954, Val Accuracy: 0.7661
{}


100%|██████████| 938/938 [00:12<00:00, 77.58it/s]


...

Epoch 14/64, Train Loss: 0.4510, Val Loss: 0.4799, Val Accuracy: 0.8362
{}


100%|██████████| 938/938 [00:39<00:00, 23.67it/s]


Epoch 15/64, Train Loss: 0.4462, Val Loss: 0.4758, Val Accuracy: 0.8364
{}


 43%|████▎     | 406/938 [00:18<00:24, 22.03it/s]


KeyboardInterrupt: 

In [ ]:
wandb.finish()

len,▁█
len_choose,▁▁
max,▁▁
sum,█▁
train_loss,██▃▂▂▂▁▁▁▁▁█▃▂▂▂▁▁▁▁
val_accuracy,▂▁▄▅▆▆▇▆▇▇█▁▄▅▆▇▇▇█▇
val_loss,▇█▅▄▃▂▂▂▁▁▁█▅▃▃▂▂▂▁▂
len,5014
len_choose,446
max,0.1479
sum,15.81557
